In [10]:
from array import array
from collections import deque, defaultdict, Counter, namedtuple

# Student Identity
LAST_NAME = "PALAOAG"
STUDENT_ID = "TUPM-26-1949"
FAVORITE_ARTIST = "RICO BLANCO"
SEED_NUM = int(STUDENT_ID[-1])

# Identity-Based Computational Seed (immutable tuples)
LAST_NAME    = LAST_NAME.upper().replace(" ", "")
SEED_DIGITS = (SEED_NUM,)
ARTIST_CODES = tuple(ord(ch) for ch in FAVORITE_ARTIST.upper() if ch.isalpha())
IDENTITY_SEED = sum(SEED_DIGITS) * len(ARTIST_CODES) + sum(c * (i + 1) for i, c in enumerate(ARTIST_CODES))

def make_stream():
    """Fresh number generator. Two deques are rotated on every call, so the
    SEED_NUM digits and FAVORITE_ARTIST character positions keep cycling."""
    digit_ring = deque(SEED_DIGITS)
    code_ring = deque(ARTIST_CODES)
    state = {"step": 0}

    def next_value(lo, hi):
        step = state["step"]
        pos = step % len(code_ring) + 1        
        n = (digit_ring[0] + 1) * code_ring[0] * pos + code_ring[-1] + step * 7 + IDENTITY_SEED % 13
        digit_ring.rotate(-1)
        code_ring.rotate(-1)
        state["step"] += 1
        return lo + n % (hi - lo + 1)

    return next_value

def table(columns, rows):
    line = " | ".join(f"{title:<{w}}" for title, w in columns)
    print(line)
    print("-" * len(line))
    for r in rows:
        print(" | ".join(f"{str(v):<{w}}" for v, (_, w) in zip(r, columns)))

def mean(values):
    return sum(values) / len(values) if len(values) else 0.0

print("IDENTITY_SEED =", IDENTITY_SEED)
print("SEED_DIGITS   =", SEED_DIGITS)
print("ARTIST_CODES  =", ARTIST_CODES)

IDENTITY_SEED = 4093
SEED_DIGITS   = (9,)
ARTIST_CODES  = (82, 73, 67, 79, 66, 76, 65, 78, 67, 79)


In [ ]:
Transaction = namedtuple("Transaction", "customer product qty")   

customers = tuple(f"{LAST_NAME}_C{i}" for i in range(1, 6))
PRICES = {"NOTEBOOK": 45.00, "PEN": 12.50, "USB": 350.00, "HEADPHONES": 899.00, "CALCULATOR": 520.00}
products = tuple(PRICES)

def value(t):
    return round(t.qty * PRICES.get(t.product, 0.0), 2)           

nxt = make_stream()
transactions = [Transaction(c, products[(i + j * 2) % 5], nxt(1, 9))
                for i, c in enumerate(customers) for j in range(2)]
transactions.append(Transaction(customers[0], transactions[0].product, nxt(1, 9)))   
transactions.append(Transaction(customers[2], transactions[4].product, nxt(1, 9)))   

table([("ID", 3), ("Customer", 14), ("Product", 11), ("Qty", 4), ("Price", 8), ("Total", 10)],
      [(n, t.customer, t.product, t.qty, f"{PRICES[t.product]:.2f}", f"{value(t):.2f}")
       for n, t in enumerate(transactions, 1)])

ID  | Customer       | Product     | Qty  | Price    | Total     
-----------------------------------------------------------------
1   | PALAOAG_C1     | NOTEBOOK    | 2    | 45.00    | 90.00     
2   | PALAOAG_C1     | USB         | 4    | 350.00   | 1400.00   
3   | PALAOAG_C2     | PEN         | 3    | 12.50    | 37.50     
4   | PALAOAG_C2     | HEADPHONES  | 2    | 899.00   | 1798.00   
5   | PALAOAG_C3     | USB         | 8    | 350.00   | 2800.00   
6   | PALAOAG_C3     | CALCULATOR  | 2    | 520.00   | 1040.00   
7   | PALAOAG_C4     | HEADPHONES  | 9    | 899.00   | 8091.00   
8   | PALAOAG_C4     | NOTEBOOK    | 3    | 45.00    | 135.00    
9   | PALAOAG_C5     | CALCULATOR  | 2    | 520.00   | 1040.00   
10  | PALAOAG_C5     | PEN         | 5    | 12.50    | 62.50     
11  | PALAOAG_C1     | NOTEBOOK    | 9    | 45.00    | 405.00    
12  | PALAOAG_C3     | USB         | 2    | 350.00   | 700.00    


In [12]:
organized = {}
for t in transactions:
    organized.setdefault(t.customer, {}).setdefault(t.product, []).append(t.qty)

for cust_name, items in organized.items():
    print(cust_name)
    for prod_name, qtys in items.items():
        print(f"   {prod_name:<11}: quantities {qtys}, value {sum(qtys) * PRICES[prod_name]:.2f}")

PALAOAG_C1
   NOTEBOOK   : quantities [2, 9], value 495.00
   USB        : quantities [4], value 1400.00
PALAOAG_C2
   PEN        : quantities [3], value 37.50
   HEADPHONES : quantities [2], value 1798.00
PALAOAG_C3
   USB        : quantities [8, 2], value 3500.00
   CALCULATOR : quantities [2], value 1040.00
PALAOAG_C4
   HEADPHONES : quantities [9], value 8091.00
   NOTEBOOK   : quantities [3], value 135.00
PALAOAG_C5
   CALCULATOR : quantities [2], value 1040.00
   PEN        : quantities [5], value 62.50


In [ ]:
pair_counts = Counter((t.customer, t.product) for t in transactions)
repeated = {k: n for k, n in pair_counts.items() if n > 1}

for (c, p), n in repeated.items():
    print(f"Repeated: {c} bought {p} {n} times")

distinct_customers = list(dict.fromkeys(t.customer for t in transactions))   
distinct_products  = list(dict.fromkeys(t.product for t in transactions))
print("Distinct customers:", len(distinct_customers))
print("Distinct products :", len(distinct_products))
print("Distinct customer-product pairs:", len(set(pair_counts)))

# Mathematical set operations
bought = {c: {t.product for t in transactions if t.customer == c} for c in customers}
a, b = bought[customers[0]], bought[customers[2]]
print(f"\nProducts of {customers[0]}: {sorted(a)}")
print(f"Products of {customers[2]}: {sorted(b)}")
print("Union                :", sorted(a | b))
print("Intersection         :", sorted(a & b))
print("Difference (A - B)   :", sorted(a - b))
print("Symmetric difference :", sorted(a ^ b))
print("Products never sold  :", sorted(set(PRICES) - {t.product for t in transactions}))

Repeated: PALAOAG_C1 bought NOTEBOOK 2 times
Repeated: PALAOAG_C3 bought USB 2 times
Distinct customers: 5
Distinct products : 5
Distinct customer-product pairs: 10

Products of PALAOAG_C1: ['NOTEBOOK', 'USB']
Products of PALAOAG_C3: ['CALCULATOR', 'USB']
Union                : ['CALCULATOR', 'NOTEBOOK', 'USB']
Intersection         : ['USB']
Difference (A - B)   : ['NOTEBOOK']
Symmetric difference : ['CALCULATOR', 'NOTEBOOK']
Products never sold  : []


In [14]:
values = array("d", [value(t) for t in transactions])
grand_total = round(sum(values), 2)
print(f"Highest transaction  : {max(values):.2f}")
print(f"Lowest transaction   : {min(values):.2f}")
print(f"Average transaction  : {mean(values):.2f}")
print(f"Overall dataset total: {grand_total:.2f}")

Highest transaction  : 8091.00
Lowest transaction   : 37.50
Average transaction  : 1466.58
Overall dataset total: 17599.00


In [15]:
cust = defaultdict(lambda: {"tx": 0, "qty": 0, "value": 0.0})
prod = defaultdict(lambda: {"tx": 0, "qty": 0, "value": 0.0})
for t in transactions:
    for bucket, key in ((cust, t.customer), (prod, t.product)):
        bucket[key]["tx"] += 1
        bucket[key]["qty"] += t.qty
        bucket[key]["value"] += value(t)

table([("Customer", 14), ("Transactions", 12), ("Items", 6), ("Total Spent", 12)],
      [(c, cust[c]["tx"], cust[c]["qty"], f"{cust[c]['value']:.2f}") for c in customers])

Customer       | Transactions | Items  | Total Spent 
-----------------------------------------------------
PALAOAG_C1     | 3            | 15     | 1895.00     
PALAOAG_C2     | 2            | 5      | 1835.50     
PALAOAG_C3     | 3            | 12     | 4540.00     
PALAOAG_C4     | 2            | 12     | 8226.00     
PALAOAG_C5     | 2            | 7      | 1102.50     


In [16]:
empty = {"tx": 0, "qty": 0, "value": 0.0}
table([("Product", 11), ("Price", 8), ("Transactions", 12), ("Units Sold", 10), ("Revenue", 10)],
      [(p, f"{PRICES[p]:.2f}", d["tx"], d["qty"], f"{d['value']:.2f}")
       for p in PRICES for d in [prod.get(p, empty)]])

Product     | Price    | Transactions | Units Sold | Revenue   
---------------------------------------------------------------
NOTEBOOK    | 45.00    | 3            | 14         | 630.00    
PEN         | 12.50    | 2            | 8          | 100.00    
USB         | 350.00   | 3            | 14         | 4900.00   
HEADPHONES  | 899.00   | 2            | 11         | 9889.00   
CALCULATOR  | 520.00   | 2            | 4          | 2080.00   


In [17]:
def rank_by_value(summary, title, width):
    queue = deque(sorted(summary.items(), key=lambda kv: kv[1]["value"], reverse=True))
    rows, rank = [], 1
    while queue:
        name, d = queue.popleft()
        rows.append((rank, name, f"{d['value']:.2f}"))
        rank += 1
    table([("Rank", 4), (title, width), ("Value", 10)], rows)
    return rows

print("Customers by total spent")
cust_rank = rank_by_value(cust, "Customer", 14)
print("\nProducts by revenue")
prod_rank = rank_by_value(prod, "Product", 11)

Customers by total spent
Rank | Customer       | Value     
----------------------------------
1    | PALAOAG_C4     | 8226.00   
2    | PALAOAG_C3     | 4540.00   
3    | PALAOAG_C1     | 1895.00   
4    | PALAOAG_C2     | 1835.50   
5    | PALAOAG_C5     | 1102.50   

Products by revenue
Rank | Product     | Value     
-------------------------------
1    | HEADPHONES  | 9889.00   
2    | USB         | 4900.00   
3    | CALCULATOR  | 2080.00   
4    | NOTEBOOK    | 630.00    
5    | PEN         | 100.00    


In [18]:
units = array("i", [t.qty for t in transactions])
most_units = max(prod, key=lambda p: prod[p]["qty"])
print(f"Total transactions     : {len(transactions)}")
print(f"Total units sold       : {sum(units)}")
print(f"Total sales value      : {grand_total:.2f}")
print(f"Top customer           : {cust_rank[0][1]} ({cust_rank[0][2]})")
print(f"Top product by revenue : {prod_rank[0][1]} ({prod_rank[0][2]})")
print(f"Most units sold        : {most_units} ({prod[most_units]['qty']} units)")
print(f"Repeated purchases     : {len(repeated)}")

Total transactions     : 12
Total units sold       : 51
Total sales value      : 17599.00
Top customer           : PALAOAG_C4 (8226.00)
Top product by revenue : HEADPHONES (9889.00)
Most units sold        : NOTEBOOK (14 units)
Repeated purchases     : 2
